# Supervised Learning

Notebook analisis kode untuk klasifikasi, regresi, regularisasi, ensemble, SVM, neural network, dan estimasi ketidakpastian. Seluruh eksperimen memakai `random_state` agar dapat diulang.

## Persiapan dan impor pustaka

Kode berikut memakai API scikit-learn yang masih didukung. Dataset Boston yang sudah dihapus dari scikit-learn diganti otomatis dengan dataset diabetes bawaan apabila tidak tersedia.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles, load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (8, 5)


## 1. Dataset sintetis dan pola evaluasi

Forge digunakan untuk klasifikasi dua kelas, sedangkan wave digunakan untuk regresi dengan hubungan nonlinear.

In [ ]:
X_forge, y_forge = make_blobs(n_samples=26, centers=2, n_features=2,
                              cluster_std=1.2, random_state=RANDOM_STATE)
x_wave = np.linspace(-3, 3, 40).reshape(-1, 1)
y_wave = np.sin(x_wave[:, 0]) + 0.15 * x_wave[:, 0] + 0.15 * np.random.randn(40)

X_train, X_test, y_train, y_test = train_test_split(
    X_forge, y_forge, test_size=0.25, random_state=RANDOM_STATE, stratify=y_forge)
knn_clf = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
print("KNN classification accuracy:", round(knn_clf.score(X_test, y_test), 3))

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    x_wave, y_wave, test_size=0.25, random_state=RANDOM_STATE)
knn_reg = KNeighborsRegressor(n_neighbors=3).fit(Xr_train, yr_train)
print("KNN regression R2:", round(knn_reg.score(Xr_test, yr_test), 3))


## 2. Pengaruh jumlah tetangga pada KNN

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, [1, 3, 9]):
    model = KNeighborsClassifier(n_neighbors=k).fit(X_forge, y_forge)
    xx, yy = np.meshgrid(
        np.linspace(X_forge[:, 0].min()-1, X_forge[:, 0].max()+1, 250),
        np.linspace(X_forge[:, 1].min()-1, X_forge[:, 1].max()+1, 250))
    pred = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, pred, alpha=.25, cmap="coolwarm")
    ax.scatter(X_forge[:, 0], X_forge[:, 1], c=y_forge, cmap="coolwarm", edgecolor="k")
    ax.set_title(f"k = {k}")
plt.tight_layout()
plt.show()


## 3. Regresi linear, Ridge, dan Lasso

In [ ]:
linear = LinearRegression().fit(Xr_train, yr_train)
print("Linear coefficient:", linear.coef_)
print("Linear intercept:", linear.intercept_)
print("Linear train/test R2:", round(linear.score(Xr_train, yr_train), 3),
      round(linear.score(Xr_test, yr_test), 3))

# Dataset regresi bawaan sebagai pengganti load_boston pada API modern.
try:
    from sklearn.datasets import load_boston
    regression_data = load_boston()
    regression_name = "Boston"
except Exception:
    from sklearn.datasets import load_diabetes
    regression_data = load_diabetes()
    regression_name = "Diabetes replacement"

X_reg, y_reg = regression_data.data, regression_data.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=.25, random_state=RANDOM_STATE)

for alpha in [1.0, 10.0, .1]:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    ridge.fit(X_reg_train, y_reg_train)
    print(f"Ridge alpha={alpha}: train={ridge.score(X_reg_train,y_reg_train):.3f}, "
          f"test={ridge.score(X_reg_test,y_reg_test):.3f}")

for alpha in [.01, .001, .0001]:
    lasso = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=100000))
    lasso.fit(X_reg_train, y_reg_train)
    print(f"Lasso alpha={alpha}: train={lasso.score(X_reg_train,y_reg_train):.3f}, "
          f"test={lasso.score(X_reg_test,y_reg_test):.3f}")
print("Regression dataset:", regression_name, X_reg.shape)


## 4. Model linear untuk klasifikasi dan Naive Bayes

In [ ]:
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=.25, random_state=RANDOM_STATE, stratify=cancer.target)
for C in [0.01, 1, 100]:
    logreg = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    logreg.fit(X_train, y_train)
    print(f"Logistic C={C}: train={logreg.score(X_train,y_train):.3f}, "
          f"test={logreg.score(X_test,y_test):.3f}")

linear_svc = make_pipeline(StandardScaler(), LinearSVC(C=1, dual="auto", max_iter=5000))
linear_svc.fit(X_train, y_train)
print("Linear SVC test accuracy:", round(linear_svc.score(X_test, y_test), 3))

X_count = np.array([[0,1,0,1],[1,0,1,1],[0,0,0,1],[1,0,1,0]])
y_count = np.array([0,1,0,1])
nb = MultinomialNB().fit(X_count, y_count)
print("Naive Bayes prediction:", nb.predict([[1,1,0,0]]))


## 5. Multiclass classification

In [ ]:
X_multi, y_multi = make_blobs(n_samples=120, centers=3, n_features=2,
                             cluster_std=1.2, random_state=RANDOM_STATE)
svc_multi = LinearSVC(dual="auto", max_iter=5000).fit(X_multi, y_multi)
print("Coefficient shape:", svc_multi.coef_.shape)
print("Intercept shape:", svc_multi.intercept_.shape)


## 6. Decision tree dan feature importance

In [ ]:
tree = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
shallow_tree = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Tree train/test:", round(tree.score(X_train,y_train),3), round(tree.score(X_test,y_test),3))
print("Depth 4 train/test:", round(shallow_tree.score(X_train,y_train),3), round(shallow_tree.score(X_test,y_test),3))
importance = sorted(zip(cancer.feature_names, shallow_tree.feature_importances_), key=lambda z:z[1], reverse=True)
print("Top features:", importance[:5])

reg_tree = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE).fit(Xr_train, yr_train)
print("Tree regression test R2:", round(reg_tree.score(Xr_test, yr_test), 3))


## 7. Ensemble: Random Forest dan Gradient Boosting

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
forest.fit(X_train, y_train)
gbrt = GradientBoostingClassifier(n_estimators=100, learning_rate=.05, max_depth=1, random_state=RANDOM_STATE)
gbrt.fit(X_train, y_train)
for name, model in [("Random Forest", forest), ("Gradient Boosting", gbrt)]:
    print(name, "train/test:", round(model.score(X_train,y_train),3), round(model.score(X_test,y_test),3))


## 8. SVM nonlinear, kernel trick, dan scaling

In [ ]:
X_moon, y_moon = make_moons(n_samples=200, noise=.2, random_state=RANDOM_STATE)
for model in [SVC(kernel="linear"), SVC(kernel="rbf", C=10, gamma="scale")]:
    model.fit(X_moon, y_moon)
    print(model.kernel, "accuracy:", round(model.score(X_moon, y_moon), 3))

svc_raw = SVC(C=10).fit(cancer.data, cancer.target)
svc_scaled = make_pipeline(StandardScaler(), SVC(C=10)).fit(X_train, y_train)
print("SVC raw/scaled test:", round(svc_raw.score(cancer.data, cancer.target),3),
      round(svc_scaled.score(X_test,y_test),3))


## 9. Neural network dan estimasi ketidakpastian

In [ ]:
mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(30,), alpha=1,
                                                    max_iter=1000, random_state=RANDOM_STATE))
mlp.fit(X_train, y_train)
print("MLP train/test:", round(mlp.score(X_train,y_train),3), round(mlp.score(X_test,y_test),3))

iris = load_iris()
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    iris.data, iris.target, test_size=.25, random_state=RANDOM_STATE, stratify=iris.target)
prob_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, multi_class="auto"))
prob_model.fit(Xi_train, yi_train)
print("Decision function shape:", prob_model.decision_function(Xi_test).shape)
print("Probability shape:", prob_model.predict_proba(Xi_test).shape)
print("Test accuracy:", round(prob_model.score(Xi_test, yi_test),3))


## Ringkasan

Notebook ini mengikuti alur `fit`, `predict`, `score`, evaluasi data latih dan data uji, visualisasi, regularisasi, scaling, serta estimasi ketidakpastian. Nilai keluaran dapat berubah jika versi pustaka atau pembagian data diubah.